In [1]:
import os
from getpass import getpass
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import random

from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split

SEED = 42


In [2]:
# Check to see if files are downloaded before running the following cell
!ls -la data

ls: cannot access 'data': No such file or directory


In [3]:
# Get Kaggle API Token
os.environ['KAGGLE_API_TOKEN'] = getpass("Kaggle API token: ")

# Download kaggle competition datasets
!kaggle competitions download -c llm-classification-finetuning

# Unzip the files
!unzip -q llm-classification-finetuning.zip -d data

# Lists the files
!ls -la data

100% 57.0M/57.0M [00:02<00:00, 22.0MB/s]

total 179884
drwxr-xr-x 2 root root      4096 Jul 23 21:35 .
drwxr-xr-x 1 root root      4096 Jul 23 21:35 ..
-rw-r--r-- 1 root root       237 Oct  9  2024 sample_submission.csv
-rw-r--r-- 1 root root     10658 Oct  9  2024 test.csv
-rw-r--r-- 1 root root 184175071 Oct  9  2024 train.csv


# Archetecure of model:

### input variables
```
prompt = prompt
response_1 = either reponse_a or response_b
response_2 = the other response_a/b
```
### Encoding model
```
model_1 = DeBERTA v3 base
```

### Classification model
```
model_2 = classification model
```
### pipeline of data
```
prompt -> model_1 -> output_p
response_1 -> model_1 -> output_1
response_2 -> model_1 -> output_2

(output_p, output_1, output_2) -> model_2 -> (prob of first response, prob of second response, prob of tie)
```



In [8]:
class ResponsePreferenceDataset(Dataset):
    """
    A custom Dataset class for handling prompts, responses from two models, and preference labels.
    This Dataset uses lazy loading.
    """
    def __init__(self, prompts, responses_a, responses_b, labels, tokenizer, max_len=512, prob=0.5):
        """
        Initializes the dataset.

        Args:
            prompts (list): A list of strings containing the prompts.
            responses_a (list): A list of strings containing the responses to
                model_a.
            responses_b (list): A list of strings containing the responses to
                model_b.
            labels (ndarray): A 1-d array of integers containing the preferred
                response.
                - 0: response_a preferred
                - 1: response_b preferred
                - 2: tie
            tokenizer (object): An object used to convert text into numerical
                indices.
            max_len (int): The maximum sequence length allowed for each
                prompt/response.
            prob (float): The probability (between 0 and 1) of permuting the
                order of responses.
        """
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.prob = prob
        self.prompts = prompts
        self.responses_a = responses_a
        self.responses_b = responses_b
        self.labels = labels

    def __len__(self):
        """
        Returns the number of samples in the dataset.

        Returns:
            int: The number of samples in the dataset.
        """
        return len(self.prompts)

    def __getitem__(self, idx):
        """
        Retrieves and tokenizes one sample, randomly swaps response_a/b (with
            probability self.prob) to reduce positional bias in training.

        Args:
            idx (int): Index of the sample to retrieve.

        Returns:
            dict: A dictionary containing:
                - prompt (Tensor): Token IDs for the prompt.
                - prompt_mask (Tensor): Attention mask for the prompt.
                - response_0 (Tensor): Token IDs for response_a (possibly
                    swapped).
                - response_0_mask (Tensor): Attention mask for response_0.
                - response_1 (Tensor): Token IDs for response_b (possibly
                    swapped).
                - response_1_mask (Tensor): Attention mask for response_1.
                - label (Tensor): Integer class label 0/1/2,
                    adjusted if swapped.
        """
        prompt = self.prompts[idx]
        label = self.labels[idx]
        response_a = self.responses_a[idx]
        response_b = self.responses_b[idx]

        # Swap responses and adjust label with probability self.prob.
        if random.random() < self.prob:
            response_a, response_b = response_b, response_a
            if label == 0:
                label = 1
            elif label ==1:
                label = 0

        # Switch from response_a/response_b to response_0/response_1 since
        # response_a and response_b may have been swapped.

        response_0, response_1 = response_a, response_b

        encoded_p = self.tokenizer(prompt, max_length=self.max_len,
                                   truncation=True, padding="max_length",
                                   return_tensors="pt")
        encoded_0 = self.tokenizer(response_0, max_length=self.max_len,
                                   truncation=True, padding="max_length",
                                   return_tensors="pt")
        encoded_1 = self.tokenizer(response_1, max_length=self.max_len,
                                   truncation=True, padding="max_length",
                                   return_tensors="pt")

        # After retrieving the tokens, we remove the extra dimension that was added
        # by the tokenizer (the batch dim) even for a single string.
        prompt = encoded_p["input_ids"].squeeze(0)
        response_0 = encoded_0["input_ids"].squeeze(0)
        response_1 = encoded_1["input_ids"].squeeze(0)

        prompt_mask = encoded_p["attention_mask"].squeeze(0)
        response_0_mask = encoded_0["attention_mask"].squeeze(0)
        response_1_mask = encoded_1["attention_mask"].squeeze(0)

        label = torch.tensor(label, dtype=torch.long)

        return {
            "prompt": prompt, "prompt_mask": prompt_mask,
            "response_0": response_0, "response_0_mask": response_0_mask,
            "response_1": response_1, "response_1_mask": response_1_mask,
            "label": label
        }




In [9]:

df = pd.read_csv("data/train.csv")
#df = df.head(200)

# Convert one hot to 0,1,2
labels_one_hot = df[["winner_model_a","winner_model_b","winner_tie"]]
labels = labels_one_hot.values.argmax(axis = 1)


In [6]:


tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

In [10]:
train_df, val_df, train_labels, val_labels = train_test_split(
    df,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=SEED
)

val_df, test_df, val_labels, test_labels = train_test_split(
    val_df,
    val_labels,
    test_size=0.5,
    stratify=val_labels,
    random_state=SEED
)

In [11]:
train_dataset = ResponsePreferenceDataset(
    prompts = train_df["prompt"].tolist(),
    responses_a = train_df["response_a"].tolist(),
    responses_b = train_df["response_b"].tolist(),
    labels= train_labels,
    tokenizer = tokenizer
)
val_dataset = ResponsePreferenceDataset(
    prompts = val_df["prompt"].tolist(),
    responses_a = val_df["response_a"].tolist(),
    responses_b = val_df["response_b"].tolist(),
    labels= val_labels,
    tokenizer = tokenizer,
    prob=0
)
test_dataset = ResponsePreferenceDataset(
    prompts = test_df["prompt"].tolist(),
    responses_a = test_df["response_a"].tolist(),
    responses_b = test_df["response_b"].tolist(),
    labels= test_labels,
    tokenizer = tokenizer,
    prob=0
)

In [12]:
batch_size = 32
train_loader = DataLoader( train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader( val_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader( test_dataset, batch_size=batch_size, shuffle=True)

In [13]:
model_1 = AutoModel.from_pretrained("microsoft/deberta-v3-base", torch_dtype=torch.float32)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  371MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
batch = next(iter(test_loader))

model_1.eval()
with torch.no_grad():
    output_p = model_1(input_ids=batch["prompt"][:1], attention_mask=batch["prompt_mask"][:1]).last_hidden_state[:,0,:]
    output_0 = model_1(input_ids=batch["response_0"][:1], attention_mask=batch["response_0_mask"][:1]).last_hidden_state[:,0,:]
    output_1 = model_1(input_ids=batch["response_1"][:1], attention_mask=batch["response_1_mask"][:1]).last_hidden_state[:,0,:]
print(output_p.shape, output_0.shape, output_1.shape)

torch.Size([1, 768]) torch.Size([1, 768]) torch.Size([1, 768])


In [17]:
class PreferenceClassifier(nn.Module):
    """
    Classification head that takes pooled encoder representations of the prompt,
    response_0, and response_1 and predicts which response is preferred:
        - 0 response_0 preferred
        - 1 response_1 preferred
        - 2 tie
    """
    def __init__(self, hidden_state=768, num_classes=3):
        """
        Initializes the classifier.

        Args:
            - hidden_state (int): The dimension of the encoding of each text.
            - num_classes (int): The number of classes.
        """
        super().__init__()
        self.fc1 = nn.Linear(hidden_state*3, hidden_state)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.fc2 = nn.Linear(hidden_state, num_classes)

    def forward(self, output_p, output_0, output_1):
        """
        Executes the forward pass of the model to generate classification
        logits.

        Args:
            - output_p (Tensor): Encoded prompt.
            - output_0 (Tensor): Encoded response_0.
            - output_1 (Tensor): Encoded response_1.

        Returns:
            Tensor: Classification logits.
        """
        combined = torch.cat([output_p, output_0, output_1], dim=1)
        x = self.fc1(combined)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.fc2(x)

        return logits


In [18]:
model_2 = PreferenceClassifier()
logits = model_2(output_p, output_0, output_1)
print(logits.shape)
print(logits)

torch.Size([1, 3])
tensor([[ 0.0508,  0.2322, -0.0992]], grad_fn=<AddmmBackward0>)


In [25]:
print(next(model_1.parameters()).dtype)


torch.float32
